# AndinaLog 03B | Eventos de flota | Diagnóstico

El notebook aplana el JSON sin alterar sus valores, aplica reglas técnicas y de negocio y genera tres salidas reproducibles.

## Contrato de diagnóstico

- Las fechas sin zona explícita representan hora de Bolivia.
- Solo las copias posteriores y los incumplimientos críticos pasan a cuarentena.
- `N/D` y `reconocido` faltante se documentan sin inventar datos.
- `valor_lectura` no tiene una unidad común declarada; se conserva para trazabilidad.

In [ ]:
from pathlib import Path
import json
import re
import sys
import pandas as pd

ENTORNO = "auto"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"

def encontrar_raiz():
    if ENTORNO == "drive" or (ENTORNO == "auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
        if not (raiz / "datasets/AndinaLog_03B_Bronce/andinalog_flota_eventos.json").is_file():
            raise FileNotFoundError(raiz)
        return raiz
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets/AndinaLog_03B_Bronce/andinalog_flota_eventos.json").is_file():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz de practicasNotebookColab")

RAIZ = encontrar_raiz()
RUTA_BRONZE = RAIZ / "datasets/AndinaLog_03B_Bronce/andinalog_flota_eventos.json"

SALIDAS = RAIZ / "proyecto-integrador/01_diagnostico/andinalog_flota_eventos/salidas"
TIPOS = {"EXCESO_VELOCIDAD", "ALERTA_TEMP_CADENA_FRIO", "MANTENIMIENTO_PROGRAMADO", "FALLA_MOTOR", "GEOCERCA_SALIDA"}
SEVERIDADES = {"Baja", "Media", "Alta"}

doc = json.loads(RUTA_BRONZE.read_text(encoding="utf-8-sig"))
filas = []
for camion in doc.get("camiones", []):
    cfg = camion.get("config", {})
    for evento in camion.get("eventos", []):
        filas.append({
            "sistema": doc.get("sistema"), "fecha_exportacion": doc.get("fecha_exportacion"),
            "camion_id": camion.get("camion_id"), "evento_id": evento.get("evento_id"),
            "timestamp": evento.get("timestamp"), "tipo": evento.get("tipo"),
            "severidad": evento.get("severidad"), "valor_lectura": evento.get("valor_lectura"),
            "reconocido": evento.get("reconocido", pd.NA),
            "umbral_temp_cabina_c": cfg.get("umbral_temp_cabina_c"),
            "geocerca_radio_km": cfg.get("geocerca_radio_km"),
            "ultimo_mantenimiento": cfg.get("ultimo_mantenimiento")})
bronze = pd.DataFrame(filas)
bronze.insert(0, "fila_bronze", range(1, len(bronze) + 1))
CAMPOS = [c for c in bronze.columns if c != "fila_bronze"]
df = bronze.copy(deep=True)
for c in CAMPOS:
    df[f"{c}_en_cuarentena"] = False
    df[f"{c}_motivo"] = ""

def marcar(campo, mascara, motivo, cuarentena=True):
    mascara = pd.Series(mascara, index=df.index).fillna(False).astype(bool)
    previo = df.loc[mascara, f"{campo}_motivo"]
    df.loc[mascara, f"{campo}_motivo"] = previo.where(previo.eq(""), previo + "; ") + motivo
    if cuarentena:
        df.loc[mascara, f"{campo}_en_cuarentena"] = True

texto = lambda c: df[c].astype("string").str.strip()
marcar("sistema", texto("sistema").ne("TELEMATICA-AndinaLog"), "Sistema distinto del contrato")
exportacion = pd.to_datetime(df["fecha_exportacion"], errors="coerce")
marcar("fecha_exportacion", exportacion.isna(), "Fecha de exportación inválida")
marcar("camion_id", ~texto("camion_id").str.fullmatch(r"CAM-\d{2}").fillna(False), "Formato esperado: CAM-##")
marcar("evento_id", ~texto("evento_id").str.fullmatch(r"EVT-ALOG-\d{5}").fillna(False), "Formato esperado: EVT-ALOG-#####")
marcar("evento_id", texto("evento_id").duplicated(keep="first"), "Copia exacta posterior del evento")

ts_iso = pd.to_datetime(df["timestamp"], format="ISO8601", errors="coerce")
ts_local = pd.to_datetime(df["timestamp"], format="%d/%m/%Y %H:%M", errors="coerce")
ts = ts_iso.fillna(ts_local)
marcar("timestamp", ts.isna(), "Fecha y hora inválida")
marcar("timestamp", ts.notna() & exportacion.notna() & ts.gt(exportacion), "Evento posterior a la exportación")
marcar("tipo", ~texto("tipo").isin(TIPOS), "Tipo fuera del catálogo permitido")
marcar("severidad", ~texto("severidad").isin(SEVERIDADES), "Severidad fuera del catálogo: Baja, Media o Alta")

valor_txt = texto("valor_lectura")
valor_num = pd.to_numeric(valor_txt, errors="coerce")
marcar("valor_lectura", valor_txt.eq("N/D"), "Lectura no disponible; conservar como nulo analítico", cuarentena=False)
marcar("valor_lectura", valor_txt.ne("N/D") & valor_num.isna(), "Lectura no numérica distinta de N/D")
marcar("valor_lectura", valor_num.notna(), "Unidad no declarada; interpretar según el tipo de evento", cuarentena=False)

rec_valido = df["reconocido"].isin([True, False]) | df["reconocido"].isna()
marcar("reconocido", ~rec_valido, "Valor distinto de verdadero, falso o nulo")
marcar("reconocido", df["reconocido"].isna(), "Estado de reconocimiento no informado", cuarentena=False)

umbral = pd.to_numeric(df["umbral_temp_cabina_c"], errors="coerce")
radio = pd.to_numeric(df["geocerca_radio_km"], errors="coerce")
manto = pd.to_datetime(df["ultimo_mantenimiento"], errors="coerce")
marcar("umbral_temp_cabina_c", umbral.isna() | ~umbral.between(0, 15), "Umbral térmico fuera del rango contractual 0 a 15 °C")
marcar("geocerca_radio_km", radio.isna() | ~radio.between(0.1, 200), "Radio de geocerca fuera del rango contractual (0, 200] km")
marcar("ultimo_mantenimiento", manto.isna(), "Fecha de mantenimiento inválida")
marcar("ultimo_mantenimiento", manto.notna() & exportacion.notna() & manto.gt(exportacion), "Mantenimiento posterior a la exportación")

flags = [f"{c}_en_cuarentena" for c in CAMPOS]
df["en_cuarentena"] = df[flags].any(axis=1)
columnas = ["fila_bronze", *CAMPOS] + [x for c in CAMPOS for x in (f"{c}_en_cuarentena", f"{c}_motivo")] + ["en_cuarentena"]
diagnosticado = df[columnas].copy()
cuarentena = diagnosticado.loc[diagnosticado["en_cuarentena"]].copy()
metricas = {
    "filas_bronze": len(bronze), "filas_diagnosticadas": len(diagnosticado),
    "filas_cuarentena": len(cuarentena), "eventos_unicos": int((~texto("evento_id").duplicated(keep="first")).sum()),
    "copias_exactas_posteriores": int(texto("evento_id").duplicated(keep="first").sum()),
    "valor_lectura_nd": int(valor_txt.eq("N/D").sum()),
    "reconocido_faltante": int(df["reconocido"].isna().sum())}
reporte = pd.DataFrame([{"metrica": k, "valor": v} for k, v in metricas.items()])
assert len(diagnosticado) == 192
assert len(cuarentena) == 8
assert metricas["eventos_unicos"] == 184
SALIDAS.mkdir(parents=True, exist_ok=True)
base = "andinalog_flota_eventos_"
diagnosticado.to_csv(SALIDAS / f"{base}diagnosticado.csv", index=False, encoding="utf-8-sig")
cuarentena.to_csv(SALIDAS / f"{base}cuarentena.csv", index=False, encoding="utf-8-sig")
reporte.to_csv(SALIDAS / f"{base}reporte_calidad.csv", index=False, encoding="utf-8-sig")
print(metricas)
print(diagnosticado.head(3).to_string())


## Resultado esperado

192 filas diagnosticadas, 184 eventos únicos y 8 copias exactas posteriores en cuarentena.